In [3]:
import duckdb 
import pandas as pd 
from urllib.request import urlopen
import json
import pandas as pd

In [24]:
# API call to get weather data
response1 = urlopen('https://api.openf1.org/v1/weather?')
weather_json = json.loads(response1.read().decode('utf-8'))
weather_df = pd.DataFrame(weather_json)

In [25]:
#API call to get race_control data
response2 = urlopen('https://api.openf1.org/v1/race_control?')
race_control_json = json.loads(response2.read().decode('utf-8'))
race_control_df = pd.DataFrame(race_control_json)

In [28]:
#API call to get meeting data
response3 = urlopen('https://api.openf1.org/v1/meetings?')
meetings_json = json.loads(response3.read().decode('utf-8'))
meetings_df = pd.DataFrame(meetings_json)

In [29]:
#API call to get starting grid data
response4 = urlopen('https://api.openf1.org/v1/starting_grid?')
starting_grid_json = json.loads(response4.read().decode('utf-8'))
starting_grid_df = pd.DataFrame(starting_grid_json)

In [ ]:
#API call to get intervals data
#Uses sample URL (session_key=9165) for schema design
response5 = urlopen('https://api.openf1.org/v1/intervals?session_key=9165')
intervals_json = json.loads(response5.read().decode('utf-8'))
intervals_df = pd.DataFrame(intervals_json)

In [54]:
#API call to get car_data data
#Uses sample URL (session_key=9159) for schema design
response6 = urlopen('https://api.openf1.org/v1/car_data?session_key=9159&speed<=315')
car_data_json = json.loads(response6.read().decode('utf-8'))
car_data_df = pd.DataFrame(car_data_json)

In [ ]:
# Connect to (or create) your local DuckDB database file
# This creates openf1_warehouse.duckdb in your OpenF1DataWarehouse folder 
con = duckdb.connect('openf1_warehouse.duckdb') 
 
# Drop table if it exists, then load sessions.csv directly into DuckDB
con.execute("DROP TABLE IF EXISTS raw_sessions") 

con.execute(""" CREATE TABLE raw_sessions AS 
            SELECT * FROM read_csv_auto('C:/Users/melvi/Documents/OpenF1DataWarehouse/RawData/sessions.csv') """)

# Verify row count
print(con.execute("SELECT count(*) as total_rows FROM raw_sessions").fetchdf())

   total_rows
0         490


In [13]:
# Check column types
print(con.execute("DESCRIBE raw_sessions").fetchdf())

           column_name               column_type null   key default extra
0          circuit_key                    BIGINT  YES  None    None  None
1   circuit_short_name                   VARCHAR  YES  None    None  None
2         country_code                   VARCHAR  YES  None    None  None
3          country_key                    BIGINT  YES  None    None  None
4         country_name                   VARCHAR  YES  None    None  None
5             date_end  TIMESTAMP WITH TIME ZONE  YES  None    None  None
6           date_start  TIMESTAMP WITH TIME ZONE  YES  None    None  None
7           gmt_offset                   VARCHAR  YES  None    None  None
8             location                   VARCHAR  YES  None    None  None
9          meeting_key                    BIGINT  YES  None    None  None
10         session_key                    BIGINT  YES  None    None  None
11        session_name                   VARCHAR  YES  None    None  None
12        session_type                

In [21]:
# Load session_result data
con.execute("DROP TABLE IF EXISTS raw_session_result") 

con.execute(""" CREATE TABLE raw_session_result AS 
            SELECT * FROM read_csv_auto('RawData/session_result.csv') """)

print(con.execute("SELECT count(*) as total_session_results FROM raw_session_result").fetchdf())
print(con.execute("SELECT * FROM raw_session_result LIMIT 140").fetchdf())

   total_session_results
0                   7572
       dnf    dns  driver_number    dsq  duration gap_to_leader  meeting_key  \
0    False  False             24  False     91.61             0         1140   
1    False  False              1  False     91.65          0.04         1140   
2    False  False             14  False    92.205         0.595         1140   
3    False  False             21  False    92.222         0.612         1140   
4    False  False             27  False    92.466         0.856         1140   
..     ...    ...            ...    ...       ...           ...          ...   
135  False  False             14  False  5675.373        38.637         1141   
136  False  False             55  False  5684.788        48.052         1141   
137  False  False             44  False  5687.713        50.977         1141   
138  False  False             18  False  5691.238        54.502         1141   
139  False  False             63  False  5692.609        55.873       

In [14]:
# Check column types
print(con.execute("DESCRIBE raw_session_result").fetchdf())

       column_name column_type null   key default extra
0              dnf     BOOLEAN  YES  None    None  None
1              dns     BOOLEAN  YES  None    None  None
2    driver_number      BIGINT  YES  None    None  None
3              dsq     BOOLEAN  YES  None    None  None
4         duration     VARCHAR  YES  None    None  None
5    gap_to_leader     VARCHAR  YES  None    None  None
6      meeting_key      BIGINT  YES  None    None  None
7   number_of_laps      BIGINT  YES  None    None  None
8           points      DOUBLE  YES  None    None  None
9         position      BIGINT  YES  None    None  None
10     session_key      BIGINT  YES  None    None  None


In [11]:
# Load weather data from API
con.execute("DROP TABLE IF EXISTS raw_weather") 

con.execute(""" CREATE TABLE raw_weather AS 
            SELECT * FROM weather_df """)

print(con.execute("SELECT count(*) as total_weather_data FROM raw_weather").fetchdf())
print(con.execute("SELECT * FROM raw_weather LIMIT 5").fetchdf())

   total_weather_data
0               43928
                               date  session_key  humidity  pressure  \
0  2023-02-24T06:51:58.804000+00:00         7763      49.0    1011.7   
1  2023-02-24T06:52:58.803000+00:00         7763      49.0    1011.7   
2  2023-02-24T06:53:58.802000+00:00         7763      50.0    1011.7   
3  2023-02-24T06:54:58.816000+00:00         7763      50.0    1011.7   
4  2023-02-24T06:55:58.815000+00:00         7763      50.0    1011.7   

   rainfall  track_temperature  wind_speed  meeting_key  wind_direction  \
0         0               29.5         4.0         1140             183   
1         0               29.6         5.0         1140             192   
2         0               29.6         3.9         1140             190   
3         0               29.7         3.9         1140             175   
4         0               29.7         4.0         1140             171   

   air_temperature  
0             24.0  
1             23.9  
2        

In [15]:
# Check column types
print(con.execute("DESCRIBE raw_weather").fetchdf())

         column_name column_type null   key default extra
0               date     VARCHAR  YES  None    None  None
1        session_key      BIGINT  YES  None    None  None
2           humidity      DOUBLE  YES  None    None  None
3           pressure      DOUBLE  YES  None    None  None
4           rainfall      BIGINT  YES  None    None  None
5  track_temperature      DOUBLE  YES  None    None  None
6         wind_speed      DOUBLE  YES  None    None  None
7        meeting_key      BIGINT  YES  None    None  None
8     wind_direction      BIGINT  YES  None    None  None
9    air_temperature      DOUBLE  YES  None    None  None


In [18]:
# Load race_control data from API
con.execute("DROP TABLE IF EXISTS raw_race_control") 

con.execute(""" CREATE TABLE raw_race_control AS 
            SELECT * FROM race_control_df """)

print(con.execute("SELECT count(*) as total_race_control_data FROM raw_race_control").fetchdf())
print(con.execute("SELECT * FROM raw_race_control LIMIT 10").fetchdf())

   total_race_control_data
0                    18686
   meeting_key  session_key                              date  driver_number  \
0         1140         7763         2023-02-24T06:45:11+00:00            NaN   
1         1140         7763         2023-02-24T06:57:05+00:00            NaN   
2         1140         7763         2023-02-24T07:00:00+00:00            NaN   
3         1140         7763  2023-02-24T07:00:00.217000+00:00            NaN   
4         1140         7763         2023-02-24T11:00:03+00:00            NaN   
5         1140         7763         2023-02-24T11:00:06+00:00            NaN   
6         1140         7763         2023-02-24T11:00:28+00:00            NaN   
7         1140         7763         2023-02-24T11:03:10+00:00            NaN   
8         1140         7763  2023-02-24T11:03:10.173000+00:00            NaN   
9         1140         7763         2023-02-24T11:05:16+00:00            NaN   

   lap_number       category           flag   scope  sector  qual

In [19]:
# Check column types
print(con.execute("DESCRIBE raw_race_control").fetchdf())

         column_name column_type null   key default extra
0        meeting_key      BIGINT  YES  None    None  None
1        session_key      BIGINT  YES  None    None  None
2               date     VARCHAR  YES  None    None  None
3      driver_number      DOUBLE  YES  None    None  None
4         lap_number      DOUBLE  YES  None    None  None
5           category     VARCHAR  YES  None    None  None
6               flag     VARCHAR  YES  None    None  None
7              scope     VARCHAR  YES  None    None  None
8             sector      DOUBLE  YES  None    None  None
9   qualifying_phase      DOUBLE  YES  None    None  None
10           message     VARCHAR  YES  None    None  None


In [ ]:
# Load meetings data from API
con.execute("DROP TABLE IF EXISTS raw_meetings") 

con.execute(""" CREATE TABLE raw_meetings AS 
            SELECT * FROM meetings_df """)

print(con.execute("SELECT count(*) as total_meetings_data FROM raw_meetings").fetchdf())
print(con.execute("SELECT * FROM raw_meetings LIMIT 10").fetchdf())

   total_meetings_data
0                  100
   meeting_key               meeting_name  \
0         1140         Pre-Season Testing   
1         1141         Bahrain Grand Prix   
2         1142   Saudi Arabian Grand Prix   
3         1143      Australian Grand Prix   
4         1207      Azerbaijan Grand Prix   
5         1208           Miami Grand Prix   
6         1209  Emilia Romagna Grand Prix   
7         1210          Monaco Grand Prix   
8         1211         Spanish Grand Prix   
9         1212        Canadian Grand Prix   

                               meeting_official_name   location  country_key  \
0           FORMULA 1 ARAMCO PRE-SEASON TESTING 2023     Sakhir           36   
1         FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2023     Sakhir           36   
2        FORMULA 1 STC SAUDI ARABIAN GRAND PRIX 2023     Jeddah          153   
3         FORMULA 1 ROLEX AUSTRALIAN GRAND PRIX 2023  Melbourne            5   
4               FORMULA 1 AZERBAIJAN GRAND PRIX 2023      

In [38]:
print(con.execute("DESCRIBE raw_meetings").fetchdf())

              column_name column_type null   key default extra
0             meeting_key      BIGINT  YES  None    None  None
1            meeting_name     VARCHAR  YES  None    None  None
2   meeting_official_name     VARCHAR  YES  None    None  None
3                location     VARCHAR  YES  None    None  None
4             country_key      BIGINT  YES  None    None  None
5            country_code     VARCHAR  YES  None    None  None
6            country_name     VARCHAR  YES  None    None  None
7            country_flag     VARCHAR  YES  None    None  None
8             circuit_key      BIGINT  YES  None    None  None
9      circuit_short_name     VARCHAR  YES  None    None  None
10           circuit_type     VARCHAR  YES  None    None  None
11       circuit_info_url     VARCHAR  YES  None    None  None
12          circuit_image     VARCHAR  YES  None    None  None
13             gmt_offset     VARCHAR  YES  None    None  None
14             date_start     VARCHAR  YES  None    Non

In [30]:
# Load starting_grid data
con.execute("DROP TABLE IF EXISTS raw_starting_grid") 

con.execute(""" CREATE TABLE raw_starting_grid AS 
            SELECT * FROM read_csv_auto('RawData/starting_grid.csv') """)

print(con.execute("SELECT count(*) as total_starting_grid FROM raw_starting_grid").fetchdf())
print(con.execute("SELECT * FROM raw_starting_grid LIMIT 10").fetchdf())

   total_starting_grid
0                 1822
   driver_number  lap_duration  meeting_key  position  session_key
0              1        89.708         1141         1         7768
1             11        89.846         1141         2         7768
2             16        90.000         1141         3         7768
3             55        90.154         1141         4         7768
4             14        90.336         1141         5         7768
5             63        90.340         1141         6         7768
6             44        90.384         1141         7         7768
7             18        90.836         1141         8         7768
8             31        90.984         1141         9         7768
9             27           NaN         1141        10         7768


In [39]:
print(con.execute("DESCRIBE raw_starting_grid").fetchdf())

     column_name column_type null   key default extra
0  driver_number      BIGINT  YES  None    None  None
1   lap_duration      DOUBLE  YES  None    None  None
2    meeting_key      BIGINT  YES  None    None  None
3       position      BIGINT  YES  None    None  None
4    session_key      BIGINT  YES  None    None  None


In [32]:
# Load laps data
con.execute("DROP TABLE IF EXISTS raw_laps") 

con.execute(""" CREATE TABLE raw_laps AS 
            SELECT * FROM read_csv_auto('RawData/laps.csv') """)

print(con.execute("SELECT count(*) as total_laps FROM raw_laps").fetchdf())
print(con.execute("SELECT * FROM raw_laps LIMIT 10").fetchdf())

   total_laps
0      214991
  date_start  driver_number  duration_sector_1  duration_sector_2  \
0        NaT             14                NaN             53.322   
1        NaT             34                NaN             51.057   
2        NaT             63                NaN             47.609   
3        NaT             23                NaN             51.682   
4        NaT             18                NaN             42.285   
5        NaT             77                NaN             46.452   
6        NaT              4                NaN             55.608   
7        NaT             22                NaN             52.812   
8        NaT             21                NaN             34.815   
9        NaT             18                NaN             30.784   

   duration_sector_3  i1_speed  i2_speed  is_pit_out_lap  lap_duration  \
0             27.970       215       200            True           NaN   
1             32.123       149       239            True        

In [40]:
print(con.execute("DESCRIBE raw_laps").fetchdf())

          column_name               column_type null   key default extra
0          date_start  TIMESTAMP WITH TIME ZONE  YES  None    None  None
1       driver_number                    BIGINT  YES  None    None  None
2   duration_sector_1                    DOUBLE  YES  None    None  None
3   duration_sector_2                    DOUBLE  YES  None    None  None
4   duration_sector_3                    DOUBLE  YES  None    None  None
5            i1_speed                    BIGINT  YES  None    None  None
6            i2_speed                    BIGINT  YES  None    None  None
7      is_pit_out_lap                   BOOLEAN  YES  None    None  None
8        lap_duration                    DOUBLE  YES  None    None  None
9          lap_number                    BIGINT  YES  None    None  None
10        meeting_key                    BIGINT  YES  None    None  None
11  segments_sector_1                   VARCHAR  YES  None    None  None
12  segments_sector_2                   VARCHAR  YE

In [33]:
# Load position data
con.execute("DROP TABLE IF EXISTS raw_position") 

con.execute(""" CREATE TABLE raw_position AS 
            SELECT * FROM read_csv_auto('RawData/position.csv') """)

print(con.execute("SELECT count(*) as total_position FROM raw_position").fetchdf())
print(con.execute("SELECT * FROM raw_position LIMIT 10").fetchdf())

   total_position
0          282708
                              date  driver_number  meeting_key  position  \
0 2023-02-24 01:51:09.680000-05:00              1         1140         1   
1 2023-02-24 01:51:09.680000-05:00              2         1140         2   
2 2023-02-24 01:51:09.680000-05:00              4         1140         3   
3 2023-02-24 01:51:09.680000-05:00             10         1140         4   
4 2023-02-24 01:51:09.680000-05:00             11         1140         5   
5 2023-02-24 01:51:09.680000-05:00             14         1140         6   
6 2023-02-24 01:51:09.680000-05:00             16         1140         7   
7 2023-02-24 01:51:09.680000-05:00             20         1140         8   
8 2023-02-24 01:51:09.680000-05:00             21         1140         9   
9 2023-02-24 01:51:09.680000-05:00             22         1140        10   

   session_key  
0         7763  
1         7763  
2         7763  
3         7763  
4         7763  
5         7763  
6       

In [41]:
print(con.execute("DESCRIBE raw_position").fetchdf())

     column_name               column_type null   key default extra
0           date  TIMESTAMP WITH TIME ZONE  YES  None    None  None
1  driver_number                    BIGINT  YES  None    None  None
2    meeting_key                    BIGINT  YES  None    None  None
3       position                    BIGINT  YES  None    None  None
4    session_key                    BIGINT  YES  None    None  None


In [37]:
# Load stints data
con.execute("DROP TABLE IF EXISTS raw_stints") 

con.execute(""" CREATE TABLE raw_stints AS 
            SELECT * FROM read_csv_auto('RawData/stints.csv') """)

print(con.execute("SELECT count(distinct session_key) as distinct_sessions FROM raw_stints").fetchdf())
print(con.execute("SELECT * FROM raw_stints LIMIT 10").fetchdf())

   distinct_sessions
0                380
       compound  driver_number  lap_end  lap_start  meeting_key  session_key  \
0        MEDIUM             14        3          1         1140         7763   
1        MEDIUM             24        3          1         1140         7763   
2        MEDIUM              4        2          1         1140         7763   
3  TEST_UNKNOWN             55        6          1         1140         7763   
4        MEDIUM             22        4          1         1140         7763   
5        MEDIUM             20        2          1         1140         7763   
6        MEDIUM             31        5          1         1140         7763   
7        MEDIUM              4        3          2         1140         7763   
8  TEST_UNKNOWN             11        4          1         1140         7763   
9          SOFT              2        3          1         1140         7763   

   stint_number  tyre_age_at_start  
0             1                  0  
1  

In [42]:
print(con.execute("DESCRIBE raw_stints").fetchdf())

         column_name column_type null   key default extra
0           compound     VARCHAR  YES  None    None  None
1      driver_number      BIGINT  YES  None    None  None
2            lap_end      BIGINT  YES  None    None  None
3          lap_start      BIGINT  YES  None    None  None
4        meeting_key      BIGINT  YES  None    None  None
5        session_key      BIGINT  YES  None    None  None
6       stint_number      BIGINT  YES  None    None  None
7  tyre_age_at_start      BIGINT  YES  None    None  None


In [36]:
# Load pit data
con.execute("DROP TABLE IF EXISTS raw_pit") 

con.execute(""" CREATE TABLE raw_pit AS 
            SELECT * FROM read_csv_auto('RawData/pit.csv') """)

print(con.execute("SELECT count(*) as total_pit FROM raw_pit").fetchdf())
print(con.execute("SELECT * FROM raw_pit LIMIT 10").fetchdf())

   total_pit
0      26477
                              date  driver_number  lane_duration  lap_number  \
0 2023-06-02 07:30:03.731000-04:00             14            NaN           1   
1 2023-06-02 07:30:07.637000-04:00             11            NaN           1   
2 2023-06-02 07:30:10.059000-04:00             18            NaN           1   
3 2023-06-02 07:30:11.903000-04:00             21            NaN           1   
4 2023-06-02 07:30:14.028000-04:00             31            NaN           1   
5 2023-06-02 07:30:15.575000-04:00             55            NaN           1   
6 2023-06-02 07:30:16.606000-04:00             63            NaN           1   
7 2023-06-02 07:30:19.121000-04:00             77            NaN           1   
8 2023-06-02 07:30:20.887000-04:00             22            NaN           1   
9 2023-06-02 07:30:23.278000-04:00              2            NaN           1   

   meeting_key  pit_duration  session_key  stop_duration  
0         1211           NaN      

In [43]:
print(con.execute("DESCRIBE raw_pit").fetchdf())

     column_name               column_type null   key default extra
0           date  TIMESTAMP WITH TIME ZONE  YES  None    None  None
1  driver_number                    BIGINT  YES  None    None  None
2  lane_duration                    DOUBLE  YES  None    None  None
3     lap_number                    BIGINT  YES  None    None  None
4    meeting_key                    BIGINT  YES  None    None  None
5   pit_duration                    DOUBLE  YES  None    None  None
6    session_key                    BIGINT  YES  None    None  None
7  stop_duration                    DOUBLE  YES  None    None  None


In [51]:
# Load intervals data
con.execute("DROP TABLE IF EXISTS raw_intervals") 

con.execute(""" CREATE TABLE raw_intervals AS 
            SELECT * FROM intervals_df """)

print(con.execute("SELECT count(*) as total_intervals FROM raw_intervals").fetchdf())
print(con.execute("SELECT * FROM raw_intervals LIMIT 10").fetchdf())

   total_intervals
0            24439
                               date  session_key  interval  driver_number  \
0  2023-09-17T11:01:02.356000+00:00         9165     0.000             55   
1  2023-09-17T12:03:49.814000+00:00         9165       NaN             16   
2  2023-09-17T12:03:49.814000+00:00         9165     0.285             63   
3  2023-09-17T12:03:50.095000+00:00         9165     0.298              4   
4  2023-09-17T12:03:50.267000+00:00         9165     0.114             44   
5  2023-09-17T12:03:50.408000+00:00         9165     0.190             20   
6  2023-09-17T12:03:50.502000+00:00         9165     0.075             14   
7  2023-09-17T12:03:50.752000+00:00         9165     0.245             31   
8  2023-09-17T12:03:50.892000+00:00         9165     0.127             27   
9  2023-09-17T12:03:51.095000+00:00         9165       NaN              1   

   meeting_key  gap_to_leader  
0         1219          0.000  
1         1219          0.262  
2         1219    

In [52]:
print(con.execute("DESCRIBE raw_intervals").fetchdf())

     column_name column_type null   key default extra
0           date     VARCHAR  YES  None    None  None
1    session_key      BIGINT  YES  None    None  None
2       interval      DOUBLE  YES  None    None  None
3  driver_number      BIGINT  YES  None    None  None
4    meeting_key      BIGINT  YES  None    None  None
5  gap_to_leader      DOUBLE  YES  None    None  None


In [55]:
# Load car_data data
con.execute("DROP TABLE IF EXISTS raw_car_data") 

con.execute(""" CREATE TABLE raw_car_data AS 
            SELECT * FROM car_data_df """)

print(con.execute("SELECT count(*) as total_car_data FROM raw_car_data").fetchdf())
print(con.execute("SELECT * FROM raw_car_data LIMIT 10").fetchdf())

   total_car_data
0          369395
                               date  session_key  meeting_key  driver_number  \
0  2023-09-15T12:45:02.318000+00:00         9159         1219              1   
1  2023-09-15T12:45:02.318000+00:00         9159         1219              2   
2  2023-09-15T12:45:02.318000+00:00         9159         1219              4   
3  2023-09-15T12:45:02.318000+00:00         9159         1219             10   
4  2023-09-15T12:45:02.318000+00:00         9159         1219             11   
5  2023-09-15T12:45:02.318000+00:00         9159         1219             14   
6  2023-09-15T12:45:02.318000+00:00         9159         1219             16   
7  2023-09-15T12:45:02.318000+00:00         9159         1219             18   
8  2023-09-15T12:45:02.318000+00:00         9159         1219             20   
9  2023-09-15T12:45:02.318000+00:00         9159         1219             22   

   speed  n_gear  drs  throttle  brake  rpm  
0      0       0    0         0      

In [56]:
print(con.execute("DESCRIBE raw_car_data").fetchdf())

     column_name column_type null   key default extra
0           date     VARCHAR  YES  None    None  None
1    session_key      BIGINT  YES  None    None  None
2    meeting_key      BIGINT  YES  None    None  None
3  driver_number      BIGINT  YES  None    None  None
4          speed      BIGINT  YES  None    None  None
5         n_gear      BIGINT  YES  None    None  None
6            drs      BIGINT  YES  None    None  None
7       throttle      BIGINT  YES  None    None  None
8          brake      BIGINT  YES  None    None  None
9            rpm      BIGINT  YES  None    None  None


In [57]:
# Load drivers data
con.execute("DROP TABLE IF EXISTS raw_drivers") 

con.execute(""" CREATE TABLE raw_drivers AS 
            SELECT * FROM read_csv_auto('RawData/drivers.csv') """)

print(con.execute("SELECT count(*) as total_drivers FROM raw_drivers").fetchdf())
print(con.execute("SELECT * FROM raw_drivers LIMIT 100").fetchdf())

   total_drivers
0           9951
   broadcast_name country_code  driver_number first_name        full_name  \
0    M VERSTAPPEN          NED              1        Max   Max VERSTAPPEN   
1      L SARGEANT          USA              2      Logan   Logan SARGEANT   
2        L NORRIS          GBR              4      Lando     Lando NORRIS   
3         P GASLY          FRA             10     Pierre     Pierre GASLY   
4         S PEREZ          MEX             11     Sergio     Sergio PEREZ   
..            ...          ...            ...        ...              ...   
95       L NORRIS          GBR              4      Lando     Lando NORRIS   
96        P GASLY          FRA             10     Pierre     Pierre GASLY   
97        S PEREZ          MEX             11     Sergio     Sergio PEREZ   
98       F ALONSO          ESP             14   Fernando  Fernando ALONSO   
99      C LECLERC          MON             16    Charles  Charles LECLERC   

                                         

In [58]:
print(con.execute("DESCRIBE raw_drivers").fetchdf())

       column_name column_type null   key default extra
0   broadcast_name     VARCHAR  YES  None    None  None
1     country_code     VARCHAR  YES  None    None  None
2    driver_number      BIGINT  YES  None    None  None
3       first_name     VARCHAR  YES  None    None  None
4        full_name     VARCHAR  YES  None    None  None
5     headshot_url     VARCHAR  YES  None    None  None
6        last_name     VARCHAR  YES  None    None  None
7      meeting_key      BIGINT  YES  None    None  None
8     name_acronym     VARCHAR  YES  None    None  None
9      session_key      BIGINT  YES  None    None  None
10     team_colour     VARCHAR  YES  None    None  None
11       team_name     VARCHAR  YES  None    None  None


In [59]:
# Load overtakes data
con.execute("DROP TABLE IF EXISTS raw_overtakes") 

con.execute(""" CREATE TABLE raw_overtakes AS 
            SELECT * FROM read_csv_auto('RawData/overtakes.csv') """)

print(con.execute("SELECT count(*) as total_overtakes FROM raw_overtakes").fetchdf())
print(con.execute("SELECT * FROM raw_overtakes LIMIT 10").fetchdf())

   total_overtakes
0            20293
                              date  meeting_key  overtaken_driver_number  \
0 2023-03-05 10:03:45.594000-05:00         1141                       63   
1 2023-03-05 10:03:45.594000-05:00         1141                       14   
2 2023-03-05 10:03:45.594000-05:00         1141                       55   
3 2023-03-05 10:03:45.594000-05:00         1141                       16   
4 2023-03-05 10:03:45.594000-05:00         1141                       11   
5 2023-03-05 10:03:45.594000-05:00         1141                        1   
6 2023-03-05 10:03:45.750000-05:00         1141                       63   
7 2023-03-05 10:03:45.750000-05:00         1141                       14   
8 2023-03-05 10:03:45.750000-05:00         1141                       55   
9 2023-03-05 10:03:45.750000-05:00         1141                       16   

   overtaking_driver_number  position  session_key  
0                        44         6         7953  
1                  

In [60]:
print(con.execute("DESCRIBE raw_overtakes").fetchdf())

                column_name               column_type null   key default extra
0                      date  TIMESTAMP WITH TIME ZONE  YES  None    None  None
1               meeting_key                    BIGINT  YES  None    None  None
2   overtaken_driver_number                    BIGINT  YES  None    None  None
3  overtaking_driver_number                    BIGINT  YES  None    None  None
4                  position                    BIGINT  YES  None    None  None
5               session_key                    BIGINT  YES  None    None  None


In [1]:
# Close connection
con.close()

NameError: name 'con' is not defined